In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets peft

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

login(token=userdata.get('HF_TOKEN'))

BASE_MODEL = "meta-llama/Llama-3.2-3B"
HF_USER = "ShahidHKhan"
HUB_MODEL_NAME = f"{HF_USER}/autopricer-lite"

tokenizer = AutoTokenizer.from_pretrained(HUB_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 fix, carried over from Day 3
    bnb_4bit_quant_type="nf4"
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", quantization_config=quant_config, dtype=torch.float16
)

fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
fine_tuned_model.eval()

test_ds = load_dataset(f"{HF_USER}/cars_lite_prompts")["test"]
print(f"Test set size: {len(test_ds)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.5 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26132 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1452 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1452 [00:00<?, ? examples/s]

Test set size: 1452


In [ ]:
import re

def predict_price(prompt, model, tokenizer, max_new_tokens=8):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    prompt_len = inputs["input_ids"].shape[1]
    generated = tokenizer.decode(output_ids[0, prompt_len:], skip_special_tokens=True)
    return generated

def extract_price(text):
    match = re.search(r"[\d,]+\.?\d*", text)
    if match:
        return float(match.group().replace(",", ""))
    return None

# Test on one example — same test_ds[0] used in the base model eval, for direct comparison
sample = test_ds[0]
raw_output = predict_price(sample["prompt"], fine_tuned_model, tokenizer)
predicted_price = extract_price(raw_output)
actual_price = float(sample["completion"])

print(f"Raw model output: {raw_output!r}")
print(f"Parsed prediction: {predicted_price}")
print(f"Actual price: {actual_price}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Raw model output: '16995.00'
Parsed prediction: 16995.0
Actual price: 15995.0


In [ ]:
from tqdm import tqdm

predictions = []
actuals = []
raw_outputs = []

for row in tqdm(test_ds):
    raw = predict_price(row["prompt"], fine_tuned_model, tokenizer)
    pred = extract_price(raw)
    predictions.append(pred)
    actuals.append(float(row["completion"]))
    raw_outputs.append(raw)

failed_parses = sum(p is None for p in predictions)
print(f"Failed to parse a price: {failed_parses} / {len(predictions)}")

100%|██████████| 1452/1452 [13:55<00:00,  1.74it/s]

Failed to parse a price: 0 / 1452


In [ ]:
import numpy as np

predictions = np.array(predictions)
actuals = np.array(actuals)

errors = np.abs(predictions - actuals)
mae = errors.mean()
median_error = np.median(errors)
rmse = np.sqrt(((predictions - actuals) ** 2).mean())

print(f"Fine-tuned model MAE: ${mae:,.2f}")
print(f"Median absolute error: ${median_error:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

# Same outlier check we needed for the base model — don't skip this
error_df_idx = np.argsort(errors)[::-1][:5]
print("\nTop 5 largest errors (checking for hallucination outliers):")
for i in error_df_idx:
    print(f"Predicted: {predictions[i]:,.2f} | Actual: {actuals[i]:,.2f} | Raw output: {raw_outputs[i]!r}")

Fine-tuned model MAE: $2,471.41
Median absolute error: $1,702.50
RMSE: $3,941.56

Top 5 largest errors (checking for hallucination outliers):
Predicted: 89,995.00 | Actual: 135,975.00 | Raw output: '89995.00'
Predicted: 149,995.00 | Actual: 116,900.00 | Raw output: '149995.00'
Predicted: 68,900.00 | Actual: 96,995.00 | Raw output: '68900.00'
Predicted: 124,995.00 | Actual: 97,500.00 | Raw output: '124995.00'
Predicted: 79,900.00 | Actual: 57,894.00 | Raw output: '79900.00'
